In [ ]:
# 第9周-Day1：BlueprintVersion — 为什么 Blueprint 是制品不是配置？
# matplotlib 中文字体配置
from matplotlib import font_manager
import matplotlib.pyplot as plt
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print(f"中文字体配置完成: {font_name}")

# 🧱 LangChat 心智模型｜第9周-Day1：BlueprintVersion — 为什么 Blueprint 是制品不是配置？

> **📌 今日核心问题：为什么 Blueprint 不是配置文件，而是制品（Artifact）？**
>
> **日期**：2026-07-27（周一）
>
> **本周主题**：Domain Deep Dive — 拆对象，理解为什么存在、边界在哪
>
> **今日对象**：BlueprintVersion（Domain Model §7.3 SC-03，ADR-005 D-2）

## 📅 学习进度

```
W1  ████████████████████ ✅ Transformer与大模型训练
W2  ████████████████████ ✅ 微调与RLHF
W3  ████████████████████ ✅ RAG与知识增强
W4  ████████████████████ ✅ 推理与思维链
W5  ████████████████████ ✅ Agent与工具使用
W6  ████████████████████ ✅ LLM Agent实战
W7  ████████████████████ ✅ 数字员工架构深化
W8  ████████████████████ ✅ LangChat 完整链路
W9  ██░░░░░░░░░░░░░░░░░░ 🔥 Domain Deep Dive (Day1/7)
W10 ░░░░░░░░░░░░░░░░░░░░ 📝 Governance 横切关注点
W11 ░░░░░░░░░░░░░░░░░░░░ 📝 Code Reality 面对代码事实
```

**进度: 9/13 周 (69.2%) | LangChat 心智模型 Day 8/28**

# 🔄 往期回顾（Week 8 全景）

## Week 8 走通了什么？

| Day | 主题 | 核心认知 |
|-----|------|----------|
| W8-D1 | 用户意图 | LangChat 不是 Agent Host，是被动受治理的企业能力平台 |
| W8-D2 | ApplicationContract | Contract 是业务治理一等对象，不是 API 文档 |
| W8-D3 | Blueprint→Compiler→IR | 10 阶段确定性 Compiler，Blueprint 是制品不是配置 |
| W8-D4 | Runtime 无状态 | 无状态手术室模型 + FrozenExecutionContext + 七字段 fallback |
| W8-D5 | Capability+Connector | Capability 是治理描述符不执行，Connector 独立治理是最大 Gap |
| W8-D6 | 完整链路图 | 10 站点 7 检查点完整链路图 + 治理热力图 |
| W8-D7 | Virtual CTO Review | 五维评分基线 7.2/10，技术债 6.5/10 最低 |

## 💡 今天与前面的关联

- W8-D3 首次遇到 Blueprint，但只从链路全景视角扫过
- 今天深入到 **BlueprintVersion 对象本身**：不可变性、内容寻址、生命周期、谱系
- W8-D7 发现 v2 制品链是最关键的 Gap → 今天验证 BlueprintVersion **已经有代码实现**

# 📚 Part 1：配置 vs 制品 — 核心问题

## 为什么 Blueprint 不是配置？

**配置（Configuration）**：你随时可以改的东西。改了立即生效，不需要评审、不需要版本号、不需要 digest。

**制品（Artifact）**：你创建后不能改的东西。它有唯一身份标识（digest），有生命周期状态，有谱系（lineage），创建前必须经过评审。

## 想象一个场景

你的数字员工"合同查询助手"今天回答了 500 个用户问题。明天有人修改了它的 Blueprint 配置文件——把 `effect_policy` 从 `read_only` 改成了 `conditional_write`。

- **配置模式**：修改立即生效，500 个用户可能突然触发写操作。没有审批、没有回滚、没有审计。
- **制品模式**：修改 = 新 BlueprintCandidate → 评审 → 新 BlueprintVersion → Build → Release Gate → Deployment。每步有审计。

**LangChat 选择了制品模式。**

# 📚 Part 2：人话解释

## 用 ERP 经验类比

| ERP 概念 | LangChat 概念 | 为什么类比 |
|---|---|---|
| 物料主数据（创建后不可改关键字段） | BlueprintVersion | 创建后有唯一编码，关键字段不可修改 |
| 工艺路线版本 | BlueprintVersion 的 version | 同一物料有多个版本 |
| 变更管理（ECR/ECN） | BlueprintCandidate 评审 | 任何变更走正式评审 |
| 物料编码 = 内容标识 | content_digest (SHA-256) | 相同内容 = 相同编码 |

## 已公证的合同文本类比

1. **起草**（BlueprintCandidate）：写合同草案，可以反复修改。草案不是正式合同。
2. **公证**（Admission + Source Review）：检查格式、引用、法律底线。不评估商业条件。
3. **存档**（BlueprintVersion）：公证完成后，赋予唯一编号（digest），存入档案室。从此不可修改。

# 📚 Part 3：ADR 依据 — ADR-005 D-2

## 评审两段式设计

| 阶段 | 执行者 | 内容 | 失败行为 |
|---|---|---|---|
| Admission Validation | 机器 | 结构完整性 + 引用合法性 + Policy floor + Self-Reference 禁止 + digest 可计算 | 拒绝，不进入 In Review |
| Source Review | 人工+规则 | 结构评审（强制）+ 合规评审（强制）+ 业务评审（可选） | 拒绝，保留历史证据 |

## 最关键的决策

> 评审**只承担结构性、引用合法、Policy floor 合规判定，不承担业务正确性判定**。

业务正确性归 ReleaseEvaluation。为什么？因为评审门槛必须**可机械判定、可重复执行**。

## 升级路径单向

```
Candidate: Draft → In Review → Promoted（升级为 BlueprintVersion）
                         → Rejected（拒绝，保留为历史证据）
```

Candidate 不存在"Pend + Resubmit 同一对象"路径。任何修改都必须产新 Candidate。

# 📚 Part 4：代码验证 — BlueprintVersion

## 文件：`/root/langchat/apps/backend/langchat/blueprint/version.py`

```python
@dataclass(frozen=True)  # ← Python 层面不可变
class BlueprintVersion:
    blueprint_id: str
    version: str
    tenant_id: str
    workspace_id: str
    application_contract_version_digest: str  # 引用 ACV
    originating_candidate_id: str             # 谱系指针
    content_digest: str                       # 内容寻址
    state: str = "active"                     # 默认 active（不是 draft！）
```

### 关键代码事实

1. `@dataclass(frozen=True)` — **冻结的数据类**，赋值操作抛 `ImmutableObjectError`
2. `state = "active"` — 创建后直接 active，**没有 Draft 状态**
3. `application_contract_version_digest` — 必须是 well-formed digest
4. `originating_candidate_id` — **谱系指针**，不允许匿名 Version

### 内容寻址（SHA-256）

```python
@property
def digest(self) -> str:
    canonical = json.dumps(payload, sort_keys=True, separators=(",", ":"))
    return "sha256:" + hashlib.sha256(canonical.encode("utf-8")).hexdigest()
```

### 生命周期前向唯一

```python
_VALID_TRANSITIONS = {
    "active": frozenset({"deprecated"}),
    "deprecated": frozenset({"retired"}),
    "retired": frozenset(),  # 终态
}
```

# 📚 Part 5：BlueprintRegistry — 非执行入口

## 文件：`/root/langchat/apps/backend/langchat/blueprint/registry.py`

```python
_FORBIDDEN_EXECUTION_METHODS = frozenset({"execute", "invoke", "run", "dispatch"})

@dataclass
class BlueprintRegistry:
    _store: dict[tuple[str, str, str, str], BlueprintVersion]
    
    def register(self, version) -> None: ...
    def get(self, tenant_id, ws_id, bp_id, version) -> BlueprintVersion | None: ...
    def list_versions(self, tenant_id, ws_id, bp_id) -> tuple[...]: ...
    
    def __post_init__(self):
        # 防御性深度：发现执行方法就崩溃！
        forbidden_present = cls_members & _FORBIDDEN_EXECUTION_METHODS
        if forbidden_present:
            raise RuntimeError(...)
```

**只有 register/get/list_versions，没有 execute/invoke/run/dispatch。** 这是 Charter §6.2 Single Canonical Execution Path 的代码级保证。

# 🏢 Part 6：商业地产映射

| LangChat 概念 | MI CRE 场景 | 对应关系 |
|---|---|---|
| BlueprintCandidate | 合同查询助手 AI 逻辑草案 | "支持自然语言查合同到期"的行为描述 |
| ApplicationContractVersion | 合同查询业务接口 | 输入：租户ID/时间段；输出：合同列表；read_only |
| **BlueprintVersion** | **合同查询助手 AI 逻辑定稿** | **经 IT 评审通过的 v1.3，不可修改** |
| Build | 编译为可执行计划 | 把逻辑描述编译为 Runtime 可加载的 IR |
| SkillRelease | 合同查询助手部署包 | 含 IR + Source Map + 依赖锁的 OCI 制品 |

## 场景：新增到期提醒功能

```
1. 产品经理在 EAC 编写新 Candidate：到期提醒逻辑
2. Admission 机器检查 ✅（结构+引用+policy）
3. Source Review 人工评审 ✅（结构+合规强制，业务建议可选）
4. 物化为 BlueprintVersion v1.3（digest=sha256:abc123...）
5. 旧版本 v1.2 不受影响，继续被已部署的 DeploymentRevision 引用
```

# ⚖️ Part 7：与传统方案比较

| 维度 | 配置模式 | 制品模式（BlueprintVersion） |
|---|---|---|
| 修改方式 | 直接编辑 | 新 Candidate → 评审 → 新 Version |
| 版本追踪 | 依赖外部（git） | 内建于对象（version + digest） |
| 回滚 | 找旧配置覆盖 | 物化新 DeploymentRevision 指向旧 digest |
| 评审 | 可选 | 强制（代码不可跳过） |
| 内容完整性 | 无法保证 | SHA-256 数学保证 |
| 谱系追踪 | 无 | originating_candidate_id |

| 平台 | AI 应用定义是什么 | 模式 |
|---|---|---|
| Dify | YAML 配置文件 | 配置模式 |
| LangChain | Python 代码 | 代码模式 |
| n8n | JSON workflow | 配置模式 |
| **LangChat** | **BlueprintVersion 制品** | **制品模式** |

LangChat 选择制品模式是因为面向**企业级生产环境**：审计、回滚、合规、多版本共存、供应链安全——这些是配置模式无法满足的硬需求。

In [ ]:
# 可视化：BlueprintVersion 生命周期与制品链
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# === 左图：Candidate → Version 生命周期 ===
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("BlueprintCandidate → BlueprintVersion\n评审与物化流程", fontsize=14, fontweight='bold')

# Candidate 状态
candidate_states = [
    (1.5, 8, "Draft", '#4ECDC4'),
    (4, 8, "In Review", '#FFA07A'),
    (6.5, 8.8, "Promoted", '#90EE90'),
    (6.5, 7.2, "Rejected", '#FFB6C1'),
]
for x, y, label, color in candidate_states:
    box = mpatches.FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, 
                                   boxstyle="round,pad=0.15", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# 箭头
ax.annotate('', xy=(3.3, 8), xytext=(2.2, 8), arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(2.75, 8.5, "提交", ha='center', fontsize=9, color='#333')
ax.annotate('', xy=(5.8, 8.8), xytext=(4.7, 8.3), arrowprops=dict(arrowstyle='->', lw=2, color='green'))
ax.text(5.3, 9.2, "评审通过", ha='center', fontsize=9, color='green')
ax.annotate('', xy=(5.8, 7.2), xytext=(4.7, 7.7), arrowprops=dict(arrowstyle='->', lw=2, color='red'))
ax.text(5.3, 6.8, "评审拒绝", ha='center', fontsize=9, color='red')

# Admission + Review 分隔
ax.annotate('Admission\n(机器判定)', xy=(4, 7), ha='center', fontsize=8, 
            style='italic', color='#555', bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8E8E8'))
ax.annotate('Source Review\n(人工+规则)', xy=(4, 6), ha='center', fontsize=8, 
            style='italic', color='#555', bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8E8E8'))

# Version 状态
version_states = [
    (2, 3, "Active", '#87CEEB'),
    (5, 3, "Deprecated", '#DAA520'),
    (8, 3, "Retired", '#C0C0C0'),
]
for x, y, label, color in version_states:
    box = mpatches.FancyBboxPatch((x-0.8, y-0.4), 1.6, 0.8, 
                                   boxstyle="round,pad=0.15", facecolor=color, edgecolor='black', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

ax.annotate('', xy=(4.2, 3), xytext=(2.8, 3), arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.annotate('', xy=(7.2, 3), xytext=(5.8, 3), arrowprops=dict(arrowstyle='->', lw=2, color='black'))
ax.text(5, 2.3, "前向唯一（不可回退）", ha='center', fontsize=9, color='#8B0000', style='italic')

# Promoted → Active 箭头
ax.annotate('', xy=(2, 3.8), xytext=(6.5, 8.4), 
            arrowprops=dict(arrowstyle='->', lw=2, color='green', connectionstyle='arc3,rad=-0.3'))
ax.text(3.5, 5.8, "物化为\nBlueprintVersion", ha='center', fontsize=9, color='green', fontweight='bold')

# 标注不可变
ax.text(5, 0.8, "🔒 BlueprintVersion 创建后不可修改（frozen=True + SHA-256）", 
        ha='center', fontsize=9, color='#8B0000', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#FFFACD', edgecolor='#8B0000'))

ax.set_xticks([])
ax.set_yticks([])

# === 右图：制品链中 BlueprintVersion 的位置 ===
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_title("制品链中 BlueprintVersion 的位置\n（★ 今天在这里）", fontsize=14, fontweight='bold')

chain = [
    (5, 9, "External Authoring\nClient", '#E8E8E8', 0.6),
    (5, 7.5, "BlueprintCandidate\n(SC-02)", '#FFA07A', 0.6),
    (5, 6, "★ BlueprintVersion ★\n(SC-03)", '#FFD700', 0.8),
    (5, 4.5, "BuildRun\n(SC-04/05)", '#87CEEB', 0.6),
    (5, 3, "ExecutionPlanIR\n(SC-06)", '#90EE90', 0.6),
    (5, 1.5, "SkillRelease v2\n(SC-13)", '#DDA0DD', 0.6),
]

for x, y, label, color, alpha in chain:
    box = mpatches.FancyBboxPatch((x-2, y-0.45), 4, 0.9, 
                                   boxstyle="round,pad=0.15", facecolor=color, 
                                   edgecolor='black', linewidth=1.5, alpha=alpha)
    ax2.add_patch(box)
    fontsize = 11 if '★' in label else 9
    weight = 'bold' if '★' in label else 'normal'
    ax2.text(x, y, label, ha='center', va='center', fontsize=fontsize, fontweight=weight)

# 箭头连接
for i in range(len(chain)-1):
    y_from = chain[i][1] - 0.45
    y_to = chain[i+1][1] + 0.45
    ax2.annotate('', xy=(5, y_to), xytext=(5, y_from), 
                arrowprops=dict(arrowstyle='->', lw=2, color='#333'))

# 标注
ax2.text(8, 6, "不可执行\n不可编辑\n不可部署", ha='center', va='center', fontsize=8, 
        color='#8B0000', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFACD', edgecolor='#8B0000'))
ax2.annotate('', xy=(7, 6), xytext=(7.5, 6), 
            arrowprops=dict(arrowstyle='->', lw=1.5, color='#8B0000'))

ax2.text(8, 4.5, "确定性\n10阶段\nCompiler", ha='center', va='center', fontsize=8, 
        color='#00008B',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E6E6FA', edgecolor='#00008B'))

ax2.text(8, 1.5, "唯一\n可部署制品", ha='center', va='center', fontsize=8, 
        color='#4B0082',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#E6E6FA', edgecolor='#4B0082'))

ax2.set_xticks([])
ax2.set_yticks([])

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第9周/w9d1_blueprint_lifecycle.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存：w9d1_blueprint_lifecycle.png")

In [ ]:
# 可视化：配置模式 vs 制品模式 对比
fig, ax = plt.subplots(figsize=(14, 6))

categories = ['版本追踪', '回滚能力', '评审门', '内容完整性\n(SHA-256)', '谱系追踪', '审计能力', '多版本共存', '供应链安全']
config_scores = [2, 1, 1, 0, 0, 1, 1, 0]
artifact_scores = [10, 9, 10, 10, 9, 10, 8, 9]

x = np.arange(len(categories))
width = 0.35

bars1 = ax.bar(x - width/2, config_scores, width, label='配置模式 (Dify/n8n)', 
               color='#FF9999', edgecolor='black', linewidth=0.8)
bars2 = ax.bar(x + width/2, artifact_scores, width, label='制品模式 (LangChat BlueprintVersion)', 
               color='#66B2FF', edgecolor='black', linewidth=0.8)

ax.set_ylabel('能力评分 (0-10)', fontsize=12)
ax.set_title('配置模式 vs 制品模式：企业级需求对比', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.legend(fontsize=10, loc='upper left')
ax.set_ylim(0, 12)
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.5, label='及格线')

# 在柱子上标数字
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第9周/w9d1_config_vs_artifact.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存：w9d1_config_vs_artifact.png")

In [ ]:
# 可视化：BlueprintVersion 三重不可变性保证
fig, ax = plt.subplots(figsize=(10, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title("BlueprintVersion 三重不可变性保证", fontsize=14, fontweight='bold')

# 中心：BlueprintVersion
center = mpatches.Circle((5, 5), 1.5, facecolor='#FFD700', edgecolor='black', linewidth=2)
ax.add_patch(center)
ax.text(5, 5.2, 'BlueprintVersion', ha='center', va='center', fontsize=12, fontweight='bold')
ax.text(5, 4.5, '不可变制品', ha='center', va='center', fontsize=10)

# 三个保障层
layers = [
    (2, 8, 'Python 层\n@dataclass(frozen=True)\n赋值 → ImmutableObjectError', '#FF6B6B'),
    (8, 8, '身份层\nSHA-256 内容寻址\n内容变 → digest 变 → 不是同一对象', '#4ECDC4'),
    (5, 1.5, '执行层\nRegistry 无 execute/invoke\n__post_init__ 自毁防御', '#95E1D3'),
]

for x, y, text, color in layers:
    box = mpatches.FancyBboxPatch((x-1.8, y-0.7), 3.6, 1.4,
                                   boxstyle="round,pad=0.2", facecolor=color, 
                                   edgecolor='black', linewidth=1.5, alpha=0.85)
    ax.add_patch(box)
    ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold')
    # 连线到中心
    ax.plot([x, 5], [y + (0.7 if y < 5 else -0.7), 5 + (-1.5 if y < 5 else 1.5) * 0.7], 
            color='black', linewidth=1.5, linestyle='--', alpha=0.5)

# 补充信息
ax.text(5, 9.5, "任意一层被绕过，其他层仍然有效（防御性深度）", 
        ha='center', fontsize=9, color='#8B0000', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='#FFFACD', edgecolor='#8B0000'))

ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.savefig('/root/learning-notebooks/第9周/w9d1_triple_immutable.png', dpi=150, bbox_inches='tight')
plt.show()
print("图表已保存：w9d1_triple_immutable.png")

# 🔍 Part 8：Gap Analysis

| 维度 | 目标态（ADR-005） | 代码现实 | Gap |
|---|---|---|---|
| 不可变性 | frozen=True + transition 创新对象 | ✅ 已实现 | 无 |
| 内容寻址 | SHA-256 over canonical JSON | ✅ 已实现 | 无 |
| 生命周期前向唯一 | active→deprecated→retired | ✅ 已实现 | 无 |
| 谱系指针 | originating_candidate_id | ✅ 已实现 | 无 |
| Admission 机器检查 | 结构+引用+Policy floor | ✅ 已实现 | 无 |
| Source Review 人工评审 | 结构+合规强制，业务可选 | 🔴 未实现 | 评审流程无代码 |
| Registry per-topic | 每个 (tenant, ws, bp) 独立 | ✅ 已实现 | 内存存储需持久化 |
| Registry 非执行入口 | 无 execute/invoke/run | ✅ 已实现 | 防御测试覆盖 |
| Build 拒绝 WorkflowSpec | validate_build_input() | ✅ 已实现 | 已验证 |
| Compiler 10 阶段 | Parse→...→Provenance | 🟡 stub | 大多为 pass-through |

**关键发现**：BlueprintVersion 核心设计**已落地**，是 v2 制品链中最成熟的部分。

# 💡 Part 9：今天多理解了什么

| 以前以为 | 现在知道 |
|---|---|
| BlueprintVersion 就是"有版本的 Blueprint" | 它是**不可变制品**，不可变性是数学保证（SHA-256 + frozen dataclass） |
| 评审就是"领导审批" | 评审是**两段式**：机器判定（Admission）+ 人工评审（Source Review）。不管业务正确性 |
| 不可变就是"不能改" | 不可变是**多维度**的：Python 层、逻辑层、身份层、执行层四重保障 |
| Registry 就是存储 | Registry 是**非执行入口**，通过 __post_init__ 自毁防御 |
| Candidate 被拒绝后可以修改重提 | ❌ Candidate 是**终态**的，修改 = 创建新 Candidate |
| Version 可以从 retired 回到 active | ❌ 生命周期**前向唯一**，不可逆 |

# 🔧 Part 10：重新设计时是否仍这样做

**会保留**：frozen=True、SHA-256 内容寻址、前向唯一生命周期、Registry 无执行方法 + 自毁防御、评审两段式 + 不检查业务正确性、谱系指针

**可能调整**：增加 Candidate withdraw 状态、Registry 用 event-sourced 持久化、Version 编号改语义化版本

# 📝 Daily Engineering Log

### 新增
- BlueprintVersion 类定义验证：frozen dataclass + SHA-256 digest + 前向唯一生命周期
- BlueprintCandidate 四态生命周期验证：Draft→In Review→Promoted|Rejected
- Admission 准入检查验证：结构+引用+Policy floor 三项，不检查业务正确性
- BlueprintRegistry 验证：per-topic 存储 + 禁止执行方法 + __post_init__ 防御
- Build 输入验证：validate_build_input() 拒绝 WorkflowSpec 模块对象

### 确认
- BlueprintVersion 是 v2 制品链中**代码实现最成熟**的对象
- `@dataclass(frozen=True)` + SHA-256 + 前向唯一生命周期 = 三重不可变性保证
- Registry 的"自毁式"防御是企业级代码典范

### 遗留
- Source Review 人工评审流程没有代码落地
- Registry 是内存存储，没有持久化机制
- Compiler 10 阶段大多为 pass-through stub

### 技术债
- Source Review 代码缺失 → 当前只有机器检查，没有人工评审门
- Registry 持久化缺失 → 生产需要数据库/对象存储后端

### 下一步
- 明天（Day2）：SkillRelease — 为什么它是唯一可部署单元？

# 📖 术语表

| 英文 | 音标 | 中文 |
|---|---|---|
| Artifact | /ˈɑːrtɪfækt/ | 制品 |
| BlueprintVersion | /ˈbluːprɪnt ˈvɜːrʒən/ | 蓝图版本 |
| BlueprintCandidate | /ˈbluːprɪnt ˈkændɪdət/ | 蓝图候选 |
| Content Addressing | /ˈkɒntent əˈdresɪŋ/ | 内容寻址 |
| Canonical | /kəˈnɒnɪk(ə)l/ | 规范的 |
| Admission | /ədˈmɪʃ(ə)n/ | 准入 |
| Source Review | /sɔːrs rɪˈvjuː/ | 源评审 |
| Lineage | /ˈlɪniɪdʒ/ | 谱系 |
| Digest | /ˈdaɪdʒest/ | 摘要 |
| Immutable | /ɪˈmjuːtəb(ə)l/ | 不可变的 |
| Forward-Only | /ˈfɔːrwərd ˈoʊnli/ | 前向唯一 |
| FrozenExecutionContext | /ˈfroʊzən ɪɡˈzekjuːʃən kɒntekst/ | 冻结执行上下文 |

# 📚 真实参考

| 参考 | 路径 |
|---|---|
| ADR-005 D-2 | `/root/langchat-docs/lanlnk/out/prd/langchat/output/review/ADR-005-Blueprint-artifact-chain-and-ApplicationContract.md` §5 |
| Domain Model §7.3 SC-03 | `.../v2-strategy/02-LangChat-v2-Target-Domain-Model.md` §7.3 |
| BlueprintVersion 代码 | `/root/langchat/apps/backend/langchat/blueprint/version.py` |
| BlueprintRegistry 代码 | `/root/langchat/apps/backend/langchat/blueprint/registry.py` |
| 不可变性测试 | `/root/langchat/apps/backend/tests/unit_tests/test_blueprint_version_immutability.py` |
| Admission 测试 | `/root/langchat/apps/backend/tests/unit_tests/test_blueprint_candidate_admission.py` |